In [23]:
import pandas as pd
import os
import io
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler




In [24]:
# 1. Load the dataset
df = pd.read_excel('../data/credit_card_raw.xls')

df.drop([0], axis=0, inplace=True)  # Drop the first row which contains metadata
df.reset_index(drop=True, inplace=True)  # Reset index after dropping the row

# 2. Drop non-predictive columns
if 'Unnamed: 0' in df.columns:
    df.drop('Unnamed: 0', axis=1, inplace=True)

print(df.head())


       X1 X2 X3 X4  X5  X6 X7  X8  X9 X10  ...    X15    X16    X17   X18  \
0   20000  2  2  1  24   2  2  -1  -1  -2  ...      0      0      0     0   
1  120000  2  2  2  26  -1  2   0   0   0  ...   3272   3455   3261     0   
2   90000  2  2  2  34   0  0   0   0   0  ...  14331  14948  15549  1518   
3   50000  2  2  1  37   0  0   0   0   0  ...  28314  28959  29547  2000   
4   50000  1  2  1  57  -1  0  -1   0   0  ...  20940  19146  19131  2000   

     X19    X20   X21   X22   X23  Y  
0    689      0     0     0     0  1  
1   1000   1000  1000     0  2000  1  
2   1500   1000  1000  1000  5000  0  
3   2019   1200  1100  1069  1000  0  
4  36681  10000  9000   689   679  0  

[5 rows x 24 columns]


In [25]:
print(df.info())
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 24 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   X1      30000 non-null  object
 1   X2      30000 non-null  object
 2   X3      30000 non-null  object
 3   X4      30000 non-null  object
 4   X5      30000 non-null  object
 5   X6      30000 non-null  object
 6   X7      30000 non-null  object
 7   X8      30000 non-null  object
 8   X9      30000 non-null  object
 9   X10     30000 non-null  object
 10  X11     30000 non-null  object
 11  X12     30000 non-null  object
 12  X13     30000 non-null  object
 13  X14     30000 non-null  object
 14  X15     30000 non-null  object
 15  X16     30000 non-null  object
 16  X17     30000 non-null  object
 17  X18     30000 non-null  object
 18  X19     30000 non-null  object
 19  X20     30000 non-null  object
 20  X21     30000 non-null  object
 21  X22     30000 non-null  object
 22  X23     30000 non-null

In [26]:
print(df.dtypes)

X1     object
X2     object
X3     object
X4     object
X5     object
X6     object
X7     object
X8     object
X9     object
X10    object
X11    object
X12    object
X13    object
X14    object
X15    object
X16    object
X17    object
X18    object
X19    object
X20    object
X21    object
X22    object
X23    object
Y      object
dtype: object


In [27]:
print(df['X3'].value_counts())

X3
2    14030
1    10585
3     4917
5      280
4      123
6       51
0       14
Name: count, dtype: int64


In [31]:
# Find nan values
print(df.isna().sum())

X1      0
X3      0
X5      0
X6      0
X7      0
X8      0
X9      0
X10     0
X11     0
X12     0
X13     0
X14     0
X15     0
X16     0
X17     0
X18     0
X19     0
X20     0
X21     0
X22     0
X23     0
y       0
X2_2    0
X4_1    0
X4_2    0
X4_3    0
dtype: int64


In [29]:
df.dropna(inplace=True)

In [30]:


# 3. Ordinal Encoding for Education (X3)
# Mapping: Graduate School (3) > University (2) > High School (1) > Others (0)
edu_mapping = {1: 3, 2: 2, 3: 1, 4: 0}
df['X3'] = df['X3'].replace({0:4, 5:4, 6:4}).map(edu_mapping)

df.rename(columns={'Y' : 'y'}, inplace=True)

# 4. One-Hot Encoding for Nominal features (Gender & Marriage)
nominal_cols = ['X2', 'X4']
df = pd.get_dummies(df, columns=nominal_cols, drop_first=True, dtype=int)

# 5. Split the Data FIRST (To prevent Data Leakage)
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['y']
)

# 6. Scaling (Fit on Train, Transform on both)
scaler = StandardScaler()

# Define truly continuous columns (excluding target 'y', Ordinal 'X3', and Dummies)
continuous_cols = [
    'X1', 'X5', 'X6', 'X7', 'X8', 'X9', 'X10', 'X11',
    'X12', 'X13', 'X14', 'X15', 'X16', 'X17',
    'X18', 'X19', 'X20', 'X21', 'X22', 'X23'
]

# Create copies to avoid SettingWithCopyWarning
train_df = train_df.copy()
test_df = test_df.copy()

# Fit ONLY on Training data
scaler.fit(train_df[continuous_cols])

# Transform both using the Training statistics
train_df[continuous_cols] = scaler.transform(train_df[continuous_cols])
test_df[continuous_cols] = scaler.transform(test_df[continuous_cols])

# 7. Save to Directory
output_dir = '../data'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

train_df.to_csv(os.path.join(output_dir, 'credit_card_preprocessed_train.csv'), index=False)
test_df.to_csv(os.path.join(output_dir, 'credit_card_preprocessed_test.csv'), index=False)

print("Preprocessing complete.")
print("Strategy: Split -> Fit Scaler on Train -> Transform Train & Test.")

/var/folders/jc/3b0b9y_n1tl_z_b6y_6_k9800000gn/T/ipykernel_3788/3952991266.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['X3'] = df['X3'].replace({0:4, 5:4, 6:4}).map(edu_mapping)


Preprocessing complete.
Strategy: Split -> Fit Scaler on Train -> Transform Train & Test.
